<a href="https://colab.research.google.com/github/2403a52270-commits/nlp/blob/main/NLP_LAB_2403A52270_Assignment_6_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LDA Using sample dataset and kaggle dataset

In [1]:
import pandas as pd
data=pd.read_excel('/content/LDA-Data.xlsx')

data_1 = pd.read_csv("/content/arxiv_data.csv")
display(data.head())

,News
0,Virat scored century in match
1,BJP won in elections
2,Bumra took 5 wicket in a match
3,Congress form state government


In [2]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

def nltk_preprocessing_pipeline(text):

    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))


    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    text = re.sub(r'<.*?>', '', text)

    text = re.sub(r'@\w+', '', text)

    text = re.sub(r'#\w+', '', text)

    text = text.lower()

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()


    tokenized_words = word_tokenize(text)

    filtered_words = [word for word in tokenized_words if word not in stop_words]

    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

    clean_summary = ' '.join(lemmatized_words)

    return clean_summary

print("NLTK preprocessing pipeline function created successfully!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


NLTK preprocessing pipeline function created successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
data['clean_News'] = data['News'].apply(nltk_preprocessing_pipeline)
data_1['clean_News'] = data_1['titles'].apply(nltk_preprocessing_pipeline)
print("\nComparison of previous clean_summaries and new clean_summaries_pipeline (first 5 rows):")
print(data[['clean_News']].head())
print(data_1[['clean_News']].head())


Comparison of previous clean_summaries and new clean_summaries_pipeline (first 5 rows):
                       clean_News
0      virat scored century match
1                    bjp election
2       bumra took 5 wicket match
3  congress form state government
                                          clean_News
0  survey semantic stereo matching semantic depth...
1  futureai guiding principle consensus recommend...
2  enforcing mutual consistency hard region semis...
3  parameter decoupling strategy semisupervised 3...
4  backgroundforeground segmentation interior sen...


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer_data = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
doc_term_matrix = count_vectorizer_data.fit_transform(data['clean_News'])

count_vectorizer_data_1 = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
doc_term_matrix_1 = count_vectorizer_data_1.fit_transform(data_1['clean_News'])

In [5]:
import pandas as pd

feature_names = count_vectorizer_data.get_feature_names_out()
feature_names_1 = count_vectorizer_data_1.get_feature_names_out()

bow_df = pd.DataFrame(doc_term_matrix.toarray(),columns=feature_names)
bow_df_1 = pd.DataFrame(doc_term_matrix_1.toarray(),columns=feature_names_1)

bow_top_10 = bow_df.head(10)
bow_top_10_1 = bow_df_1.head(10)
print(bow_top_10)
print(bow_top_10_1)

   bjp  bumra  century  congress  election  form  government  match  scored  \
0    0      0        1         0         0     0           0      1       1   
1    1      0        0         0         1     0           0      0       0   
2    0      1        0         0         0     0           0      1       0   
3    0      0        0         1         0     1           1      0       0   

   state  took  virat  wicket  
0      0     0      1       0  
1      0     0      0       0  
2      0     1      0       1  
3      1     0      0       0  
   01  05mb  09  10  100  1000  10000  100000  1000x  100k  ...  \
0   0     0   0   0    0     0      0       0      0     0  ...   
1   0     0   0   0    0     0      0       0      0     0  ...   
2   0     0   0   0    0     0      0       0      0     0  ...   
3   0     0   0   0    0     0      0       0      0     0  ...   
4   0     0   0   0    0     0      0       0      0     0  ...   
5   0     0   0   0    0     0      0     

In [6]:
from sklearn.decomposition import LatentDirichletAllocation


num_topics = 2
LDA_data = LatentDirichletAllocation(n_components=num_topics, random_state=42)
LDA_data.fit(doc_term_matrix)

LDA_data_1 = LatentDirichletAllocation(n_components=num_topics, random_state=42)
LDA_data_1.fit(doc_term_matrix_1)

LatentDirichletAllocation(n_components=2, random_state=42)

In [7]:
def display_topics(model, feature_names, num_top_words):
    for topic_idx in range(len(model.components_)):
        print(f"\nTopic {topic_idx}:")


        topic_weights = model.components_[topic_idx]


        sorted_indices = topic_weights.argsort()[::-1]


        top_indices = sorted_indices[:num_top_words]

        for idx in top_indices:
            print(feature_names[idx], end=" ")
        print()

In [8]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic from data:")
display_topics(LDA_data, count_vectorizer_data.get_feature_names_out(), num_top_words)
print(f"\nTop {num_top_words} words per topic from data_1:")
display_topics(LDA_data_1, count_vectorizer_data_1.get_feature_names_out(), num_top_words)


Top 10 words per topic from data:

Topic 0:
form government congress state election bjp match wicket bumra took 

Topic 1:
match virat century scored took bumra wicket bjp election state 

Top 10 words per topic from data_1:

Topic 0:
learning deep reinforcement representation network using neural image detection time 

Topic 1:
network image graph detection adversarial neural object 3d segmentation using 


In [9]:
document_topics = LDA_data.transform(doc_term_matrix)
document_topics_1 = LDA_data_1.transform(doc_term_matrix_1)
data['topic'] = document_topics.argmax(axis=1)
data_1['topic'] = document_topics_1.argmax(axis=1)

# LDA output with sample data

In [10]:
print("\nDataFrame with assigned topics (first 5 rows):")
print(data[['clean_News', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                       clean_News  topic
0      virat scored century match      1
1                    bjp election      0
2       bumra took 5 wicket match      1
3  congress form state government      0


# LDA output with kaggle dataset

In [11]:
print("\nDataFrame with assigned topics (first 5 rows):")
print(data_1[['clean_News', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                                          clean_News  topic
0  survey semantic stereo matching semantic depth...      1
1  futureai guiding principle consensus recommend...      0
2  enforcing mutual consistency hard region semis...      1
3  parameter decoupling strategy semisupervised 3...      0
4  backgroundforeground segmentation interior sen...      1


# NMF Using sample dataset and kaggle dataset

In [12]:
from sklearn.decomposition import NMF


num_topics = 2
NMF_data = NMF(n_components=num_topics, random_state=42)
NMF_data.fit(doc_term_matrix)

NMF_data_1 = NMF(n_components=num_topics, random_state=42)
NMF_data_1.fit(doc_term_matrix_1)

NMF(n_components=2, random_state=42)

In [13]:
def display_topics(model, feature_names, num_top_words):
    for topic_idx in range(len(model.components_)):
        print(f"\nTopic {topic_idx}:")


        topic_weights = model.components_[topic_idx]


        sorted_indices = topic_weights.argsort()[::-1]


        top_indices = sorted_indices[:num_top_words]

        for idx in top_indices:
            print(feature_names[idx], end=" ")
        print()

In [14]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic from data:")
display_topics(NMF_data, count_vectorizer_data.get_feature_names_out(), num_top_words)
print(f"\nTop {num_top_words} words per topic from data_1:")
display_topics(NMF_data_1, count_vectorizer_data_1.get_feature_names_out(), num_top_words)


Top 10 words per topic from data:

Topic 0:
match virat took scored wicket bumra century bjp election state 

Topic 1:
state form congress government election bjp took virat wicket scored 

Top 10 words per topic from data_1:

Topic 0:
learning reinforcement deep representation using image transfer unsupervised model data 

Topic 1:
network neural graph image using convolutional deep adversarial detection generative 


In [15]:
document_topics = NMF_data.transform(doc_term_matrix)
document_topics_1 = NMF_data_1.transform(doc_term_matrix_1)
data['topic'] = document_topics.argmax(axis=1)
data_1['topic'] = document_topics_1.argmax(axis=1)

# NMF OUTPUT WITH SAMPLEDATA

In [16]:
print("\nDataFrame with assigned topics (first 5 rows):")
print(data[['clean_News', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                       clean_News  topic
0      virat scored century match      0
1                    bjp election      1
2       bumra took 5 wicket match      0
3  congress form state government      1


# NMF OUTPUT WITH KAGGLE DATASET

In [17]:
print("\nDataFrame with assigned topics (first 5 rows):")
print(data_1[['clean_News', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                                          clean_News  topic
0  survey semantic stereo matching semantic depth...      1
1  futureai guiding principle consensus recommend...      1
2  enforcing mutual consistency hard region semis...      1
3  parameter decoupling strategy semisupervised 3...      1
4  backgroundforeground segmentation interior sen...      1


# NMF Using TF-IDF

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer_data = TfidfVectorizer(max_df=0.95, min_df=1, stop_words='english')
tfidf_matrix = tfidf_vectorizer_data.fit_transform(data['clean_News'])

tfidf_vectorizer_data_1 = TfidfVectorizer(max_df=0.95, min_df=1, stop_words='english')
tfidf_matrix_1 = tfidf_vectorizer_data_1.fit_transform(data_1['clean_News'])

print("Shape of TF-IDF matrix for data:", tfidf_matrix.shape)
print("Shape of TF-IDF matrix for data_1:", tfidf_matrix_1.shape)

Shape of TF-IDF matrix for data: (4, 13)
Shape of TF-IDF matrix for data_1: (51774, 23208)


In [19]:
from sklearn.decomposition import NMF

num_topics = 2

NMF_tfidf_data = NMF(n_components=num_topics, random_state=42)
NMF_tfidf_data.fit(tfidf_matrix)

NMF_tfidf_data_1 = NMF(n_components=num_topics, random_state=42)
NMF_tfidf_data_1.fit(tfidf_matrix_1)

print('NMF models fitted using TF-IDF for both datasets.')

NMF models fitted using TF-IDF for both datasets.


# NMF using tf-idf for sample data

In [20]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic from data (TF-IDF NMF):")
display_topics(NMF_tfidf_data, tfidf_vectorizer_data.get_feature_names_out(), num_top_words)


Top 10 words per topic from data (TF-IDF NMF):

Topic 0:
match bumra wicket took virat scored century state government election 

Topic 1:
election bjp form government state congress took virat wicket scored 


# NMF output using tf-idf for kaggle data

In [21]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic from data_1 (TF-IDF NMF):")
display_topics(NMF_tfidf_data_1, tfidf_vectorizer_data_1.get_feature_names_out(), num_top_words)


Top 10 words per topic from data_1 (TF-IDF NMF):

Topic 0:
network neural graph convolutional adversarial image generative detection using object 

Topic 1:
learning reinforcement deep representation unsupervised transfer model image using data 
